# Hopper D4RL Diagnostics — Guided vs Unguided

Follow-up to `hopper_d4rl_diagnostics.ipynb`. Generate trajectories **with guidance** and compare per-dim RMSE, action NLL, and trajectory quality against unguided baseline.

Uses SOPE's hopper config: `action_scale=0.5, normalize_grad=true, k_guide=1, use_neg_grad=true, ratio=0.5`

In [ ]:
import sys, os
os.environ["D4RL_SUPPRESS_IMPORT_ERROR"] = "1"
sys.path.insert(0, os.path.join(os.getcwd(), "../../third_party/sope"))

import torch
import numpy as np
import gym, d4rl
%matplotlib inline
import matplotlib.pyplot as plt

from opelab.core.policy import D4RLSACPolicy, D4RLPolicy
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.temporal import TemporalUnet

SOPE_ROOT = os.path.join(os.getcwd(), "../../third_party/sope")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STATE_DIM, ACTION_DIM = 11, 3

HOPPER_STATE_NAMES = [
    "z_pos", "angle", "thigh_angle", "leg_angle", "foot_angle",
    "z_vel", "angle_vel", "thigh_vel", "leg_vel", "foot_vel", "x_vel"
]

# Guidance hyperparams from hopper_stitch.json
GUIDANCE_PARAMS = dict(
    action_scale=0.5,
    normalize_grad=True,
    k_guide=1,
    use_neg_grad=True,
    neg_grad_scale=0.1,
    use_adaptive=False,
    use_action_grad_only=True,
    clamp=True,
    l_inf=1,
    ratio=0.5,
)

np.random.seed(42)
torch.manual_seed(42)
print(f"Imports done. Device: {DEVICE}")

## Setup: Load everything

In [ ]:
# Load policies — test with policy 0 (closest to behavior) and policy 10 (furthest)
policy_dir = os.path.join(SOPE_ROOT, "opelab/examples/d4rl/policy/hopper/dope")
policy_paths = sorted(
    [os.path.join(policy_dir, f) for f in os.listdir(policy_dir) if f.endswith(".pkl")],
    key=lambda p: int(os.path.basename(p).replace(".pkl", ""))
)
target_policies = {int(os.path.basename(p).replace(".pkl","")): D4RLSACPolicy(p).to(DEVICE) for p in policy_paths}
print(f"Loaded {len(target_policies)} target policies")

behavior_policy = D4RLPolicy("hopper-medium-v2").to(DEVICE)
print("Loaded behavior policy")

# Dataset
env = gym.make("hopper-medium-v2")
dataset = env.get_dataset()
obs, actions, rewards = dataset["observations"], dataset["actions"], dataset["rewards"]
terminals, timeouts = dataset["terminals"], dataset["timeouts"]
print(f"Dataset: {obs.shape[0]} transitions")

# Normalization
mean_s, std_s = np.mean(obs, axis=0), np.std(obs, axis=0)
mean_a, std_a = np.mean(actions, axis=0), np.std(actions, axis=0)
norm_mean = torch.tensor(np.concatenate([mean_s, mean_a]), dtype=torch.float32, device=DEVICE)
norm_std = torch.tensor(np.concatenate([std_s, std_a]), dtype=torch.float32, device=DEVICE)
normalize_fn = lambda x: (x - norm_mean) / norm_std
unnormalize_fn = lambda x: x * norm_std + norm_mean

# Extract real trajectories
trajs = []
start = 0
for i in range(len(obs)):
    if terminals[i] or timeouts[i]:
        trajs.append({"observations": obs[start:i+1], "actions": actions[start:i+1],
                       "rewards": rewards[start:i+1], "length": i+1-start})
        start = i + 1
        if len(trajs) >= 100:
            break
print(f"Extracted {len(trajs)} trajectories (avg length {np.mean([t['length'] for t in trajs]):.0f})")

# Load diffusion model
T_chunk, D_steps = 8, 256
temporal_model = TemporalUnet(horizon=T_chunk, transition_dim=STATE_DIM+ACTION_DIM,
                               dim_mults=(1,2,4,8), attention=False)
diffusion_model = GaussianDiffusion(
    model=temporal_model, horizon=T_chunk,
    observation_dim=STATE_DIM, action_dim=ACTION_DIM,
    n_timesteps=D_steps, predict_epsilon=True,
    normalizer=normalize_fn, unnormalizer=unnormalize_fn,
)
ckpt = torch.load(os.path.join(SOPE_ROOT, "opelab/examples/d4rl/models/hopper.pth"), map_location=DEVICE)
diffusion_model.load_state_dict(ckpt)
diffusion_model.to(DEVICE).eval()
print(f"Loaded diffusion model: T={T_chunk}, D={D_steps}")

## Trajectory generation: unguided + guided per target policy

Generate 10 trajectories (T_gen=200) for each condition:
- Unguided (baseline)
- Guided toward each of the 11 target policies

In [ ]:
def generate_trajectories(diffusion_model, normalize_fn, unnormalize_fn,
                          initial_states, T_chunk, T_gen=200, batch_size=10,
                          guided=False, guidance_hyperparams=None):
    """Generate full trajectories via autoregressive chunk stitching."""
    all_traj = torch.zeros(batch_size, T_gen, STATE_DIM + ACTION_DIM, device=DEVICE)
    end_idx = torch.full((batch_size,), T_gen, dtype=torch.long, device=DEVICE)
    alive = torch.ones(batch_size, dtype=torch.bool, device=DEVICE)

    full_init = torch.zeros(batch_size, STATE_DIM + ACTION_DIM, device=DEVICE)
    full_init[:, :STATE_DIM] = initial_states[:batch_size]
    cond_states = normalize_fn(full_init)[:, :STATE_DIM]

    t_written = 0
    for chunk_idx in range(T_gen // T_chunk + 2):
        if not alive.any() or t_written >= T_gen:
            break

        cond = {0: cond_states[alive]}
        n_alive = alive.sum().item()
        shape = (n_alive, T_chunk, STATE_DIM + ACTION_DIM)

        if guided and guidance_hyperparams:
            samples = diffusion_model.conditional_sample(shape=shape, cond=cond, guided=True,
                                                         **guidance_hyperparams)
        else:
            samples = diffusion_model.conditional_sample(shape=shape, cond=cond, guided=False)

        chunk_unnorm = unnormalize_fn(samples.trajectories)
        start = 0 if chunk_idx == 0 else 1

        for local_t in range(start, T_chunk):
            if t_written >= T_gen:
                break
            alive_indices = torch.where(alive)[0]
            all_traj[alive_indices, t_written] = chunk_unnorm[:, local_t]
            for bi, idx_val in enumerate(alive_indices):
                s = all_traj[idx_val, t_written, :STATE_DIM].cpu().numpy()
                if not (np.isfinite(s).all() and (np.abs(s[2:]) < 100).all()
                        and s[0] > 0.7 and abs(s[1]) < 0.2):
                    alive[idx_val] = False
                    end_idx[idx_val] = t_written
            t_written += 1

        if alive.any():
            last = samples.trajectories[:, -1, :]
            new_cond = torch.zeros(batch_size, STATE_DIM, device=DEVICE)
            ai = 0
            for i in range(batch_size):
                if alive[i]:
                    new_cond[i] = last[ai, :STATE_DIM]
                    ai += 1
            cond_states = new_cond

    return all_traj, end_idx

# Common initial states
N_TRAJS, T_GEN = 10, 200
init_states = torch.tensor(
    np.array([t["observations"][0] for t in trajs[:N_TRAJS]]),
    dtype=torch.float32, device=DEVICE)

# Generate unguided
print("Generating unguided trajectories...")
unguided_trajs, unguided_ends = generate_trajectories(
    diffusion_model, normalize_fn, unnormalize_fn, init_states, T_chunk, T_GEN, N_TRAJS)
print(f"  Done. Mean length: {unguided_ends.float().mean():.0f}")

# Generate guided for each target policy
guided_results = {}
for pi in sorted(target_policies.keys()):
    print(f"Generating guided trajectories for policy {pi}...")
    diffusion_model.policy = target_policies[pi]
    diffusion_model.behavior_policy = behavior_policy
    torch.manual_seed(42)  # same noise for fair comparison
    g_trajs, g_ends = generate_trajectories(
        diffusion_model, normalize_fn, unnormalize_fn, init_states, T_chunk, T_GEN, N_TRAJS,
        guided=True, guidance_hyperparams=GUIDANCE_PARAMS)
    guided_results[pi] = {"trajs": g_trajs, "ends": g_ends}
    print(f"  Done. Mean length: {g_ends.float().mean():.0f}")

print("\nAll trajectories generated.")

## Comparison 1: Per-Dimension RMSE — Guided vs Unguided

In [ ]:
def compute_per_dim_rmse(synth_trajs, end_indices, real_trajs, n_trajs, T_gen):
    mses = []
    for i in range(n_trajs):
        real_len = min(real_trajs[i]["length"], T_gen, end_indices[i].item())
        if real_len < 5:
            continue
        real_s = torch.tensor(real_trajs[i]["observations"][:real_len], dtype=torch.float32)
        synth_s = synth_trajs[i, :real_len, :STATE_DIM].cpu()
        mses.append((real_s - synth_s).pow(2).mean(dim=0))
    if not mses:
        return None
    return np.sqrt(torch.stack(mses).mean(dim=0).numpy())

# Unguided RMSE
unguided_rmse = compute_per_dim_rmse(unguided_trajs, unguided_ends, trajs, N_TRAJS, T_GEN)
print(f"Unguided total RMSE: {np.sqrt(np.mean(unguided_rmse**2)):.4f}")

# Guided RMSE per policy
guided_rmses = {}
for pi in sorted(guided_results.keys()):
    r = guided_results[pi]
    rmse = compute_per_dim_rmse(r["trajs"], r["ends"], trajs, N_TRAJS, T_GEN)
    if rmse is not None:
        guided_rmses[pi] = rmse
        total = np.sqrt(np.mean(rmse**2))
        print(f"Guided (policy {pi:>2d}) total RMSE: {total:.4f}  mean traj len: {r['ends'].float().mean():.0f}")

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: total RMSE per policy
ax = axes[0]
pids = sorted(guided_rmses.keys())
totals = [np.sqrt(np.mean(guided_rmses[p]**2)) for p in pids]
ax.bar([f"P{p}" for p in pids], totals, alpha=0.7, label="Guided")
ax.axhline(np.sqrt(np.mean(unguided_rmse**2)), color="red", linestyle="--", linewidth=2, label="Unguided")
ax.set_ylabel("Total State RMSE")
ax.set_xlabel("Target Policy")
ax.set_title("Total State RMSE: Guided vs Unguided")
ax.legend()

# Right: per-dim RMSE for unguided vs best/worst guided
ax = axes[1]
x = np.arange(STATE_DIM)
w = 0.25
ax.bar(x - w, unguided_rmse, w, label="Unguided", alpha=0.8)
best_pi = min(guided_rmses, key=lambda p: np.mean(guided_rmses[p]**2))
worst_pi = max(guided_rmses, key=lambda p: np.mean(guided_rmses[p]**2))
ax.bar(x, guided_rmses[best_pi], w, label=f"Guided P{best_pi} (best)", alpha=0.8)
ax.bar(x + w, guided_rmses[worst_pi], w, label=f"Guided P{worst_pi} (worst)", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(HOPPER_STATE_NAMES, rotation=45, ha="right")
ax.set_ylabel("RMSE")
ax.set_title("Per-Dimension RMSE")
ax.legend()

plt.tight_layout()
plt.show()

## Comparison 2: Action NLL — Does guidance shift actions toward target?

In [ ]:
def compute_nll(policy, synth_trajs, end_indices, n_trajs, use_extended=False):
    nlls = []
    for i in range(n_trajs):
        T = min(end_indices[i].item(), synth_trajs.shape[1])
        if T < 2:
            continue
        s = synth_trajs[i, :T, :STATE_DIM].detach()
        a = synth_trajs[i, :T, STATE_DIM:].detach()
        with torch.no_grad():
            if use_extended:
                lp = policy.log_prob_extended(s, a).sum(dim=-1)
            else:
                lp = policy.log_prob(s, a)
        nlls.append(-lp.mean().item())
    return np.mean(nlls), np.std(nlls)

# For each target policy: compute NLL of (unguided actions) vs (guided-toward-that-policy actions)
print(f"{'Policy':>8} {'Unguided NLL':>14} {'Guided NLL':>12} {'Delta':>8} {'Better?':>8}")
print("-" * 55)

nll_data = []
for pi in sorted(guided_results.keys()):
    policy = target_policies[pi]
    ug_nll, ug_std = compute_nll(policy, unguided_trajs, unguided_ends, N_TRAJS)
    g_nll, g_std = compute_nll(policy, guided_results[pi]["trajs"], guided_results[pi]["ends"], N_TRAJS)
    delta = ug_nll - g_nll
    better = "YES" if delta > 0 else "no"
    print(f"{pi:>8d} {ug_nll:>14.2f} {g_nll:>12.2f} {delta:>+8.2f} {better:>8}")
    nll_data.append({"pi": pi, "ug_nll": ug_nll, "g_nll": g_nll, "delta": delta})

# Also with log_prob_extended
print(f"\nUsing log_prob_extended:")
print(f"{'Policy':>8} {'Unguided NLL':>14} {'Guided NLL':>12} {'Delta':>8} {'Better?':>8}")
print("-" * 55)
nll_ext_data = []
for pi in sorted(guided_results.keys()):
    policy = target_policies[pi]
    ug_nll, _ = compute_nll(policy, unguided_trajs, unguided_ends, N_TRAJS, use_extended=True)
    g_nll, _ = compute_nll(policy, guided_results[pi]["trajs"], guided_results[pi]["ends"], N_TRAJS, use_extended=True)
    delta = ug_nll - g_nll
    better = "YES" if delta > 0 else "no"
    print(f"{pi:>8d} {ug_nll:>14.2f} {g_nll:>12.2f} {delta:>+8.2f} {better:>8}")
    nll_ext_data.append({"pi": pi, "ug_nll": ug_nll, "g_nll": g_nll, "delta": delta})

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in [(axes[0], nll_data, "log_prob"), (axes[1], nll_ext_data, "log_prob_extended")]:
    pids = [d["pi"] for d in data]
    ax.bar([f"P{p}" for p in pids], [d["ug_nll"] for d in data], alpha=0.6, label="Unguided")
    ax.bar([f"P{p}" for p in pids], [d["g_nll"] for d in data], alpha=0.6, label="Guided")
    ax.set_ylabel("NLL")
    ax.set_xlabel("Target Policy")
    ax.set_title(f"Action NLL ({title}): Unguided vs Guided")
    ax.legend()
plt.tight_layout()
plt.show()

## Comparison 3: Trajectory Plots — Guided vs Unguided vs Real

Compare policy 0 (closest to behavior) and policy 10 (furthest).

In [ ]:
key_dims = [0, 1, 5, 10]
key_names = ["z_pos", "angle", "z_vel", "x_vel"]
n_plot = 3

for target_pi in [0, 10]:
    fig, axes = plt.subplots(len(key_dims), 1, figsize=(14, 3*len(key_dims)), sharex=True)
    g_trajs = guided_results[target_pi]["trajs"]
    g_ends = guided_results[target_pi]["ends"]

    for di, (dim, name) in enumerate(zip(key_dims, key_names)):
        ax = axes[di]
        for i in range(n_plot):
            real_len = trajs[i]["length"]
            ug_len = min(unguided_ends[i].item(), T_GEN)
            g_len = min(g_ends[i].item(), T_GEN)
            ax.plot(trajs[i]["observations"][:real_len, dim],
                    color=f"C{i}", linewidth=1.5, alpha=0.7,
                    label=f"real_{i}" if di == 0 else None)
            ax.plot(unguided_trajs[i, :ug_len, dim].cpu().numpy(),
                    color=f"C{i}", linewidth=1.5, alpha=0.5, linestyle="--",
                    label=f"unguided_{i}" if di == 0 else None)
            ax.plot(g_trajs[i, :g_len, dim].cpu().numpy(),
                    color=f"C{i}", linewidth=1.5, alpha=0.5, linestyle=":",
                    label=f"guided_{i}" if di == 0 else None)
        ax.set_ylabel(name)
        ax.grid(True, alpha=0.3)
    axes[0].set_title(f"Policy {target_pi}: Real (solid) vs Unguided (dashed) vs Guided (dotted)")
    axes[0].legend(ncol=3, fontsize=8)
    axes[-1].set_xlabel("Timestep")
    plt.tight_layout()
    plt.show()

## Comparison 4: Mean Trajectory Length (Survival)

Does guidance cause early termination (quality-consistency tradeoff)?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
pids = sorted(guided_results.keys())
mean_lens = [guided_results[p]["ends"].float().mean().item() for p in pids]
ax.bar([f"P{p}" for p in pids], mean_lens, alpha=0.7, label="Guided")
ax.axhline(unguided_ends.float().mean().item(), color="red", linestyle="--", linewidth=2,
           label=f"Unguided ({unguided_ends.float().mean():.0f})")
ax.set_ylabel("Mean Trajectory Length")
ax.set_xlabel("Target Policy")
ax.set_title("Mean Trajectory Length: Guided vs Unguided (max=200)")
ax.legend()
ax.set_ylim(0, 210)
plt.tight_layout()
plt.show()

print(f"Unguided mean length: {unguided_ends.float().mean():.0f}")
for pi in pids:
    print(f"Guided P{pi}: mean length = {guided_results[pi]['ends'].float().mean():.0f}")